In [2]:
import pandas as pd
import os
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


def generate_source_data():
    os.makedirs('source_data', exist_ok=True)
    
    if os.path.exists('source_data/transactions.csv'):
        print("  -> Source files already exist. Skipping generation.")
        return

    with open('source_data/stores.csv', 'w') as f:
        f.write("store_id,location,region\n1,Mumbai,West\n2,Pune,West\n3,Delhi,North\n")

    products_json = """[
        {"product_id": 101, "name": "Apples", "category": "Groceries", "unit_price": 120},
        {"product_id": 102, "name": "Milk", "category": "Dairy", "unit_price": 65},
        {"product_id": 103, "name": "Bread", "category": "Bakery", "unit_price": -20} 
    ]"""
    with open('source_data/products.json', 'w') as f:
        f.write(products_json)

    with open('source_data/transactions.csv', 'w') as f:
        f.write("transaction_id,store_id,product_id,quantity,sale_date\n")
        f.write("1001,1,101,5,2026-10-01\n")
        f.write("1002,1,102,2,2026-10-01\n")
        f.write("1003,2,101,10,2026-10-02\n")
        f.write("1004,,103,1,2026-10-02\n") 

def load_to_dw(df, table_name, pk_column, engine):
    try:
        existing_keys = pd.read_sql(f"SELECT {pk_column} FROM {table_name}", engine)[pk_column].tolist()
        df_new = df[~df[pk_column].isin(existing_keys)]
    except Exception:
        df_new = df

    if not df_new.empty:
        df_new.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"  -> SUCCESS: Loaded {len(df_new)} new records into '{table_name}'.")
    else:
        print(f"  -> SUCCESS: '{table_name}' is up to date.")

def execute_pipeline():
    print("=== Starting Retail Data Warehouse ETL Pipeline ===\n")
    
    print("[1/5] DATA INGESTION: Sourcing raw data...")
    generate_source_data()
    
    df_stores = pd.read_csv('source_data/stores.csv')
    df_products = pd.read_json('source_data/products.json')
    df_sales = pd.read_csv('source_data/transactions.csv')

    print("\n[2/5] DATA CLEANING: Applying quality rules...")
    invalid_price_count = len(df_products[df_products['unit_price'] <= 0])
    df_products = df_products[df_products['unit_price'] > 0]
    
    missing_fk_count = df_sales['store_id'].isna().sum()
    df_sales = df_sales.dropna(subset=['store_id'])
    print(f"  -> Removed {invalid_price_count} invalid products and {missing_fk_count} orphaned sales records.")

    print("\n[3/5] TRANSFORMATION: Shaping data for Star Schema...")
    df_sales['sale_date'] = pd.to_datetime(df_sales['sale_date'])
    
    df_sales_enriched = df_sales.merge(df_products[['product_id', 'unit_price']], on='product_id', how='left')
    df_sales['total_amount'] = df_sales_enriched['quantity'] * df_sales_enriched['unit_price']
    
    df_sales['store_id'] = df_sales['store_id'].astype(int)

    print("\n[4/5] STORAGE: Loading into SQL Server Star Schema...")
    server_name = r'DESKTOP-P5RT5RO' 
    connection_url = URL.create(
        "mssql+pyodbc",
        host=server_name,
        database='RetailDW',
        query={"driver": "ODBC Driver 17 for SQL Server", "Trusted_Connection": "yes"}
    )
    engine = create_engine(connection_url, fast_executemany=True)

    load_to_dw(df_stores, 'Dim_Store', 'store_id', engine)
    load_to_dw(df_products, 'Dim_Product', 'product_id', engine)
    load_to_dw(df_sales, 'Fact_Sales', 'transaction_id', engine)

    print("\n[5/5] REPORTING: Generating Analytical Output...")
    report_query = """
        SELECT 
            s.location,
            p.category,
            SUM(f.quantity) as total_units_sold,
            SUM(f.total_amount) as total_revenue
        FROM Fact_Sales f
        JOIN Dim_Store s ON f.store_id = s.store_id
        JOIN Dim_Product p ON f.product_id = p.product_id
        GROUP BY s.location, p.category
        ORDER BY total_revenue DESC;
    """
    df_report = pd.read_sql(report_query, engine)
    print("\n=== EXECUTIVE REVENUE REPORT ===")
    print(df_report.to_string(index=False))
    print("================================\n")
    
    df_report.to_csv('source_data/warehouse_revenue_report.csv', index=False)
    print("Pipeline Execution Completed Successfully.")

if __name__ == "__main__":
    execute_pipeline()

=== Starting Retail Data Warehouse ETL Pipeline ===

[1/5] DATA INGESTION: Sourcing raw data...
  -> Source files already exist. Skipping generation.

[2/5] DATA CLEANING: Applying quality rules...
  -> Removed 1 invalid products and 1 orphaned sales records.

[3/5] TRANSFORMATION: Shaping data for Star Schema...

[4/5] STORAGE: Loading into SQL Server Star Schema...


c:\Users\admin\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1135.8'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


  -> SUCCESS: Loaded 3 new records into 'Dim_Store'.
  -> SUCCESS: Loaded 2 new records into 'Dim_Product'.
  -> SUCCESS: Loaded 3 new records into 'Fact_Sales'.

[5/5] REPORTING: Generating Analytical Output...

=== EXECUTIVE REVENUE REPORT ===
location  category  total_units_sold  total_revenue
    Pune Groceries                10           1200
  Mumbai Groceries                 5            600
  Mumbai     Dairy                 2            130

Pipeline Execution Completed Successfully.
